# 34 — PyCaret Model Screening for Regression (Objective 3)

**Objective.** Use PyCaret to compare the standard regression models on the existing training rows, then select one model family for Optuna tuning.

**Input.** `data/processed/regression_model_input.csv`; `feature_engine/model_features.csv`.

**Output.** `training_and_evaluation/pycaret_regression_model_screening.csv`; `training_and_evaluation/pycaret_regression_selected_model.csv`.

This notebook uses PyCaret to compare standard regression families on the existing training rows and selects one model family for hyperparameter tuning in the next notebook.

## 0. Setup

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

In [ ]:
import pandas as pd
import pycaret
from pycaret.regression import RegressionExperiment

print("PyCaret", pycaret.__version__)

## 1. Load the fixed training split

This section checks that the PyCaret comparison runs only on the 20,000 training rows created in the data preparation step.

In [ ]:
p = obj_paths(3)
target = "product_wg_ton"

data = pd.read_csv(p["processed"] / "regression_model_input.csv")
features = pd.read_csv(p["feature_engine"] / "model_features.csv")["feature"].tolist()

train = data.loc[data["split"].eq("train")].copy()
test = data.loc[data["split"].eq("test")].copy()

print("training rows:", len(train))
print("test rows kept untouched:", len(test))
print("feature count:", len(features))
print("training target summary:")
print(train[target].describe().round(2))

> **Interpretation.**
>
> - PyCaret sees the training rows only.
> - The fixed test set remains untouched for the final evaluation step.
> - The model table already contains the simple encodings chosen in the earlier data preparation step; PyCaret normalization is used only for fair model-family screening.

## 2. PyCaret model screening

**Rule fixed before the run.**

- Compare the standard regression model families named in the project scope.
- Rank by PyCaret's cross-validated R².
- Select the top-ranked model family for Optuna tuning in the next notebook.
- Keep the screening step as model-family selection, not final test evaluation.

In [ ]:
screen_train = train[features + [target]].copy()

exp = RegressionExperiment(
    target=target,
    session_id=RANDOM_STATE,
    train_size=0.8,
    fold=5,
    preprocess=True,
    normalize=True,
    verbose=False,
    n_jobs=1,
).fit(screen_train)

include_models = ["lr", "ridge", "lasso", "svm", "dt", "rf", "ada", "xgboost", "catboost"]
result = exp.compare_models(
    include=include_models,
    sort="R2",
    n_select=len(include_models),
    turbo=False,
    errors="raise",
    verbose=False,
)

leaderboard = result.leaderboard.copy()
model_names = exp.models()[["Name"]].rename(columns={"Name": "model_name"})
leaderboard = leaderboard.merge(model_names, left_on="Model", right_index=True, how="left")
ordered_cols = ["Model", "model_name"] + [c for c in leaderboard.columns if c not in ["Model", "model_name"]]
leaderboard = leaderboard[ordered_cols]
print(leaderboard.to_string(index=False))

selected = leaderboard.iloc[0]
selected_model = pd.DataFrame([
    {
        "selected_model_id": selected["Model"],
        "selected_model_name": selected["model_name"],
        "selection_metric": "R2",
        "selection_metric_value": selected["R2"],
        "pycaret_train_size_inside_training_rows": 0.8,
        "folds": 5,
        "screening_scope": "existing Objective 3 training rows only",
    }
])

save_table(leaderboard, p["train_eval"] / "pycaret_regression_model_screening.csv", index=False)
save_table(selected_model, p["train_eval"] / "pycaret_regression_selected_model.csv", index=False)

> **Interpretation.**
>
> - CatBoost ranks first with a cross-validated R² of **0.9943**, followed by XGBoost at **0.9938** and random forest at **0.9936**.
> - The gap between first and third is fewer than 0.001 R² points, meaning all three tree-based methods perform similarly on this data.
> - Linear models (Ridge, Lasso, linear regression) cluster near 0.9845, a noticeable step below the top three.
> - CatBoost is selected because it leads the ranking without requiring any additional modelling setup.

> **Decision — model selection.**
>
> - PyCaret ranks CatBoost first by R²: **0.9943**.
> - XGBoost is second at **0.9938** and random forest is third at **0.9936**.
> - CatBoost stays inside the small, standard model set and does not require a separate model by zone.
> - The next notebook tunes only CatBoost, as requested.

## 3. Save outputs

In [ ]:
# Both output files were written inside the screening cell above.
print(f"pycaret_regression_model_screening.csv  → {p['train_eval'] / 'pycaret_regression_model_screening.csv'}")
print(f"pycaret_regression_selected_model.csv   → {p['train_eval'] / 'pycaret_regression_selected_model.csv'}")

## 4. Checks

In [ ]:
screen_path = p["train_eval"] / "pycaret_regression_model_screening.csv"
selected_path = p["train_eval"] / "pycaret_regression_selected_model.csv"

check_screen = pd.read_csv(screen_path)
check_selected = pd.read_csv(selected_path)

assert len(check_screen) == 9
assert check_selected.loc[0, "selected_model_id"] == "catboost"
assert set(check_screen["Model"]) == set(include_models)
print("checks passed")

---
## Summary

- PyCaret screening is complete for Objective 3.
- The selected model family is **CatBoost**.
- The evaluation notebook tunes only CatBoost with Optuna and then evaluates it on the untouched test set.